# 02 - Data Understanding e Contrato de Dados

Avalia qualidade, cobertura temporal e conformidade mínima do dataset.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0]
labeled = ROOT / 'data' / 'processed' / 'labeled' / 'apontamentos_labeled.parquet'
df = pd.read_parquet(labeled)
print('Shape:', df.shape)
df.head(2)

Shape: (377907, 10)


,Id,Inicio,Fim,Tag,Frota,Tipo,Classe,next_critical_event_time,tte_horas,target_4h
0,23130822,2025-01-01 03:00:00+00:00,2025-01-01 04:00:00+00:00,CA0000,793-D 5S,Caminhao,Parado,NaT,NaN,0
1,23132927,2025-01-01 04:00:00+00:00,2025-01-01 05:00:00+00:00,CA0000,793-D 5S,Caminhao,Parado,NaT,NaN,0


In [2]:
# Qualidade básica
quality = {
    'linhas': len(df),
    'colunas': len(df.columns),
    'nulos_total': int(df.isna().sum().sum()),
    'duplicados': int(df.duplicated().sum())
}
quality

{'linhas': 377907, 'colunas': 10, 'nulos_total': 145994, 'duplicados': 0}

In [3]:
# Cobertura temporal
time_col = 'Fim' if 'Fim' in df.columns else 'Inicio'
ts = pd.to_datetime(df[time_col], errors='coerce', utc=True)
coverage = {
    'time_col': time_col,
    'inicio': str(ts.min()),
    'fim': str(ts.max()),
    'dias_cobertos': int((ts.max() - ts.min()).days) if ts.notna().any() else 0,
}
coverage

{'time_col': 'Fim',
 'inicio': '2025-01-01 03:03:43+00:00',
 'fim': '2025-07-01 03:00:00+00:00',
 'dias_cobertos': 180}

In [4]:
# Distribuição da variável alvo
target_col = 'target_4h'
dist = df[target_col].value_counts(dropna=False).rename_axis('classe').reset_index(name='qtd')
dist['proporcao'] = dist['qtd'] / dist['qtd'].sum()
dist

,classe,qtd,proporcao
0,0,307096,0.812623
1,1,70811,0.187377
